## Limpieza sales_lines
Objetivos del notebook: 
- Limpiar los datos de clientes provenientes de la capa bronce.
- Estandarizar y Limpiar los datos. 
- Mandar a cuarentena aquellos datos que requieren revision más profunda. 
- Cargar los datos en la capa plata. 

### Resumen Columnas
| Columna | Descripción |
|---|---|
| `order_id`            | Identificador unico de la venta                                            |
| `line_number`         | Número de línea, es decir puede existir una venta con 3 productos de línea |
| `order_ts`            | Momento de la compra (UTC)                                                 |
| `customer_id`         | Clave foránea de cliente                                                   |
| `product_id`          | Clave foránea del producto                                                 |
| `quantity`            | Cantidad de productos                                                      |
| `unit_price_local`    | Precio unitario en la modena local                                         |
| `currency`            | Moneda local                                                               |
| `discount_pct`        | Porcentaje de descuento                                                    |
| `coupon_code`         | Cupón usado                                                                |
| `status`              | Estado **de la línea**: completed, cancelled, returned, pending            |
| `payment_method`      | Método de pago                                                             |
| `channel`             | Canal de venta                                                             |
| `country`             | País de la venta                                                           |
| `shipping_cost_local` | Costo de envío **del pedido** (se repite en cada línea del pedido)         |
| `delivered_at`        | Fecha de entrega al cliente (UTC; nulo si no entregado)                    |
| `ingested_at`         | Fecha de ingesta, pero no brinda más detalle                               |
| `dlh_hash_sales_lines` | Hash de sales_lines generado por todas las columnas sin el prefijo `dlh_`     |
| `dlh_carga_id`     | Identificador de la carga a la que pertenece.             |
| `dlh_fecha_ingesta_bronce`| Fecha de ingesta en la capa bronce                 |
| `dlh_fecha_ingesta`| Fecha de ingesta en la capa plata                         |

### Librerias y datos 

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "01-funciones-compartidas"))

from pyspark.sql import functions as F, types as T, Window as W
from delta.tables import DeltaTable
from funciones_limpieza import limpiar_texto, estandarizar_columnas_str, fecha_de_corte_data, generar_hash
from funciones_conexion import obtener_ruta
from funciones_actualizacion_marcas import obtener_ventana_fechas_carga_id, actualizar_fechas_id_carga

## Recursos Adicionales

In [0]:
plata_customers = spark.table("dlh_nova_shop.plata.customers").select("customer_id")
plata_products = spark.table("dlh_nova_shop.plata.products").select("product_id")

## Ingesta de Datos

In [0]:
# 1. Lectura de la tabla de Bronce
schema = "bronce"
tabla = "sales_lines"
schema_actualizacion = "plata"
bronce_sales = spark.table(
    "dlh_nova_shop."+schema+"."+tabla
)

# 2. Valores de control de fechas y carga_id 
ultima_fecha_actualizacion, maxima_fecha_bronce,resumen_carga,ventana_carga_id, bronce_sales= obtener_ventana_fechas_carga_id(
    spark= spark,
    tabla_df = bronce_sales,
    nombre_tabla=tabla,
    nombre_schema_tabla_actualizacion= schema_actualizacion
    )
# 3. Resumen de valores 
print(f"Watermark anterior: {ultima_fecha_actualizacion}")
print(f"Fecha máxima de ingesta en bronce: {maxima_fecha_bronce}")
print(f"Registros a procesar en esta carga incremental: {bronce_sales.count()}")
print(f"Cargas a reemplazar (REPLACE WHERE): {ventana_carga_id}")

## Limpieza Ventas Online tipos de datos 

### Limpieza order_id

In [0]:
bronce_sales = bronce_sales.withColumn(
    "order_id", 
    F.upper(F.trim(F.col("order_id")))
    ).withColumn(
        "order_id",
        F.when(
            F.col("order_id").rlike(r"^O\d{8}$"),
            F.col("order_id")
        )
        .otherwise("N/A")
    )

### Limpieza line_number

In [0]:
bronce_sales = bronce_sales.withColumn(
    "line_number",
    F.abs(F.col("line_number")).cast("integer")
)

### Limpieza order_ts

In [0]:
bronce_sales = bronce_sales.withColumn(
    "order_ts",
    F.try_to_timestamp(F.col("order_ts"),F.lit("yyyy-MM-dd HH:mm:ss"))
)

### Limpieza customer_id

In [0]:
bronce_sales = bronce_sales.withColumn(
    "customer_id", 
    F.upper(F.trim(F.col("customer_id")))
    ).withColumn(
        "customer_id",
        F.when(
            F.col("customer_id").rlike(r"^C\d{6}$"),
            F.col("customer_id")
        )
        .otherwise("N/A")
    )

### Limpieza product_id

In [0]:
bronce_sales = bronce_sales.withColumn(
    "product_id", 
    F.upper(F.trim(F.col("product_id")))
    ).withColumn(
        "product_id",
        F.when(
            F.col("product_id").rlike(r"^P\d{4}$"),
            F.col("product_id")
        )
        .otherwise("N/A")
    )

### Limpieza quantity

In [0]:
bronce_sales = bronce_sales.withColumn(
    "quantity",
    F.abs(F.col("quantity")).cast("integer")
)

### Limpieza unit_price_local

In [0]:
bronce_sales = bronce_sales.withColumn(
    "unit_price_local",
    F.col("unit_price_local").cast("decimal(10,2)")
)

### Limpieza currency

In [0]:
bronce_sales  = bronce_sales.withColumn(
    "currency",
    F.upper(F.trim(F.col('currency')))
)

### Limpieza discount_pct

In [0]:
bronce_sales = bronce_sales.withColumn(
    "discount_pct",
    F.when(
        F.abs(F.col("discount_pct")) > 1,
        F.abs(F.col("discount_pct")/100).cast("decimal(10,2)")
    ).otherwise(
        F.abs(F.col("discount_pct")).cast("decimal(10,2)")
    )
)

### Limpieza coupon_code

In [0]:
bronce_sales = bronce_sales.withColumn(
    "coupon_code",
    F.upper(F.trim(F.col("coupon_code")))
)

### Limpieza status

In [0]:
bronce_sales = bronce_sales.withColumn(
    "status",
    F.initcap(F.trim(F.col("status")))
).withColumn(
    "status",
    F.when(
        F.col("status") == "Canceled",
        F.lit("Cancelled")
    ).otherwise(
        F.col("status")
    )
)

### Limpieza payment_method

In [0]:
bronce_sales = bronce_sales.withColumn(
    "payment_method",
    limpiar_texto(F.regexp_replace(F.col("payment_method"),"_"," "))
)

### Limpieza channel

In [0]:
bronce_sales = bronce_sales.withColumn(
    "channel",
    limpiar_texto(F.regexp_replace(F.col("channel"),"_"," "))
)

### Limpieza country

In [0]:
bronce_sales = bronce_sales.withColumn(
    "country",
    F.upper(F.trim(F.col("country")))
)

### Limpieza shipping_cost_local

In [0]:
bronce_sales = bronce_sales.withColumn(
    "shipping_cost_local",
    F.abs(F.col("shipping_cost_local")).cast("decimal(10,2)")   
)

### Limpieza delivered_at

In [0]:
bronce_sales = bronce_sales.withColumn(
    "delivered_at",
    F.try_to_timestamp(F.col("delivered_at"), F.lit("yyyy-MM-dd HH:mm:ss"))
)

### Limpieza ingested_at

In [0]:
bronce_sales = bronce_sales.withColumn(
    "ingested_at",
    F.try_to_timestamp(F.col("ingested_at"), F.lit("yyyy-MM-dd HH:mm:ss"))
)

### Limpieza duplicados

In [0]:
bronce_sales = bronce_sales.dropDuplicates()

## Limpieza Ventas Online Logica

In [0]:
C_sin_order_id = bronce_sales.filter(
    F.col("order_id").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("SIN order_id")
)
bronce_sales = bronce_sales.filter(
    F.col("order_id").isNotNull()
)

In [0]:
bronce_sales = bronce_sales.withColumn(
    "fecha_de_corte",
    fecha_de_corte_data()
)

### Integridada Referencial

In [0]:
C_sin_clientes_sin_productos = bronce_sales.join(
    how='leftanti',
    other=plata_customers,
    on='customer_id'
).join(
    how='leftanti',
    other=plata_products,
    on='product_id'
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("SIN cliente_id ni product_id, en sus tablas respectivas")
)

C_sin_clientes_con_productos = bronce_sales.join(
    how='leftanti',
    other=plata_customers,
    on='customer_id'
    ).join(
    how='leftsemi',
    other=plata_products,
    on='product_id'
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("SIN cliente_id EN LA TABLA CUSTOMERS, SI TIENE product_id EN LA TABLA PRODUCTS")
)

C_con_clientes_sin_productos = bronce_sales.join(
    how='leftsemi',
    other=plata_customers,
    on='customer_id'
    ).join(
    how='leftanti',
    other=plata_products,
    on='product_id'
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("CON cliente_id EN LA TABLA CUSTOMERS, SIN product_id EN LA TABLA PRODUCTS")
)

In [0]:
bronce_sales = bronce_sales.join(
    how='leftsemi',
    other=plata_customers,
    on='customer_id'
    ).join(
        how='leftsemi',
        other=plata_products,
        on= 'product_id'
    )

### Verificacion status

In [0]:
C_status_sin_fecha = bronce_sales.filter(
    (F.col("status")== "Completed")
    & (F.col("delivered_at").isNull())
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("status == Completed SIN delivered_at")
)
bronce_sales = bronce_sales.filter(
    ~(
        (F.col("status") == "Completed") 
        & (F.col("delivered_at").isNull())
    )
)


### Cuarentena por Fechas

In [0]:
C_sin_order_ts = bronce_sales.filter(
    F.col("order_ts").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("order_ts is NULL")
)

bronce_sales = bronce_sales.filter(
    F.col("order_ts").isNotNull()
)

In [0]:
C_sin_ingested_at = bronce_sales.filter(
    F.col("ingested_at").isNull()
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("ingested_at is NULL")
)

bronce_sales = bronce_sales.filter(
    F.col("ingested_at").isNotNull()
)

In [0]:
C_orden_despues_de_ingesta = bronce_sales.filter(
    F.col("order_ts") > F.col("ingested_at")
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("order_ts > ingested_at ")
)
bronce_sales = bronce_sales.filter(
    F.col("order_ts") <= F.col("ingested_at")
)

In [0]:
C_ingesta_futura = bronce_sales.filter(
    F.col("ingested_at") > F.col("fecha_de_corte")
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("order_ts > fecha_de_corte")
)
bronce_sales = bronce_sales.filter(
    F.col("ingested_at") <= F.col("fecha_de_corte")
)

In [0]:
C_delivered_at_menor_orden_ts = bronce_sales.filter(
    F.col("delivered_at") < F.col("order_ts")
).withColumn(
    "RAZON_CUARENTENA",
    F.lit("delivered_at < order_ts")
)
bronce_sales = bronce_sales.filter(
    (F.col("delivered_at").isNull())
    | (F.col("delivered_at") >= F.col("order_ts"))
)

## Exportando Datos

In [0]:
cuarentena_sales = C_sin_order_id \
    .unionByName(C_sin_clientes_sin_productos, allowMissingColumns=True) \
    .unionByName(C_sin_clientes_con_productos, allowMissingColumns=True) \
    .unionByName(C_con_clientes_sin_productos, allowMissingColumns=True) \
    .unionByName(C_status_sin_fecha, allowMissingColumns=True) \
    .unionByName(C_sin_order_ts, allowMissingColumns=True) \
    .unionByName(C_sin_ingested_at, allowMissingColumns=True) \
    .unionByName(C_orden_despues_de_ingesta, allowMissingColumns=True) \
    .unionByName(C_ingesta_futura, allowMissingColumns=True) \
    .unionByName(C_delivered_at_menor_orden_ts, allowMissingColumns=True)

# REPLACE WHERE: borra y reinserta SOLO las filas de las cargas de esta ventana.
# Si la ejecución falla a medias, la re-ejecución reemplaza las mismas filas (sin duplicados)
if ventana_carga_id is not None:
    cuarentena_sales.write \
        .mode("overwrite") \
        .format("delta") \
        .option("replaceWhere", ventana_carga_id) \
        .option("delta.feature.allowColumnDefaults", "supported") \
        .saveAsTable("dlh_nova_shop.cuarentena.sales_lines")
else:
    print("Bronce vacio: no hay registros que escribir en cuarentena")

In [0]:
columnas_hash = [
    "order_id","line_number", "order_ts", "customer_id", "product_id", "quantity", "unit_price_local", "currency","discount_pct", "coupon_code", "status", "payment_method", "channel", "country", "shipping_cost_local", "delivered_at","ingested_at"
]
plata_sales = generar_hash(bronce_sales,columnas_hash,"dlh_hash_sales_lines")
plata_sales = plata_sales.select(
    F.col("order_id"),
    F.col("line_number"),
    F.col("order_ts"),
    F.col("customer_id"),
    F.col("product_id"),
    F.col("quantity"),
    F.col("unit_price_local"),
    F.col("currency"),
    F.col("discount_pct"),
    F.col("coupon_code"),
    F.col("status"),
    F.col("payment_method"),
    F.col("channel"),
    F.col("country"),
    F.col("shipping_cost_local"),
    F.col("delivered_at"),
    F.col("ingested_at"),
    F.col("dlh_hash_sales_lines"),
    F.col("dlh_carga_id"),
    F.col("dlh_fecha_ingesta").alias("dlh_fecha_ingesta_bronce"),
    F.current_timestamp().alias("dlh_fecha_ingesta")
)

In [0]:
plata_sales = plata_sales.withColumn(
    "dedup_hash",
    F.row_number().over(
        W.partitionBy(
            F.col("dlh_hash_sales_lines")
        )
    )
).filter(
    F.col("dedup_hash") == 1
).drop("dedup_hash")

In [0]:
# REPLACE WHERE: borra y reinserta SOLO las filas de las cargas de esta ventana
if ventana_carga_id is not None:
    DeltaTable.forName(spark, "dlh_nova_shop.plata.sales_lines").alias("recur").merge(
        plata_sales.alias("incre"),
        "recur.dlh_hash_sales_lines == incre.dlh_hash_sales_lines"
    ).whenNotMatchedInsertAll().whenMatchedUpdateAll().execute()
else:
    print("Bronce vacio: no hay registros que escribir en plata")

## Actualizacion de tabla de control

In [0]:
actualizar_fechas_id_carga(
    spark= spark,
    resumen_carga= resumen_carga,
    maxima_fecha= maxima_fecha_bronce,
    ultima_fecha_actualizacion = ultima_fecha_actualizacion,
    tabla = tabla,
    schema = schema_actualizacion 
)